# 03 — Model Experiments

So sánh 4 models: **Random**, **Popularity**, **ALS (Collaborative Filtering)**, **Content-Based**.

**Nội dung:**
1. Load processed data
2. Train tất cả models
3. So sánh ranking metrics
4. Visualize kết quả
5. ALS hyperparameter tuning
6. Sample recommendations & similar items7. Log kết quả lên MLflow


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
import time
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

from src.config import settings
from src.evaluation.metrics import coverage, evaluate_model
from src.features.interaction import build_interaction_matrix
from src.features.item_features import build_item_features
from src.models.baseline import PopularityRecommender, RandomRecommender
from src.models.collaborative import ALSRecommender
from src.models.content_based import ContentBasedRecommender
from src.training.train import load_processed_data

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("Imports OK — project root:", PROJECT_ROOT)

## 1. Load Processed Data

In [ ]:
data = load_processed_data(settings.data_processed_dir)
train_df = data["train"]
val_df = data["val"]
test_df = data["test"]
movies_df = data["movies"]

n_users = int(train_df["user_idx"].max()) + 1
n_items = int(train_df["movie_idx"].max()) + 1

print(f"Train rows : {len(train_df):,}")
print(f"Val rows   : {len(val_df):,}")
print(f"Test rows  : {len(test_df):,}")
print(f"Users      : {n_users:,}")
print(f"Items      : {n_items:,}")
print(f"Sparsity   : {1 - len(train_df) / (n_users * n_items):.4%}")

In [ ]:
explicit_matrix, implicit_matrix = build_interaction_matrix(
    train_df,
    n_users,
    n_items,
    implicit_threshold=settings.implicit_threshold,
)
print(f"Interaction matrix: {implicit_matrix.shape}, nnz={implicit_matrix.nnz:,}")

## 2. Train All Models

In [ ]:
K = settings.top_k
SAMPLE_USERS = 500
THRESHOLD = settings.implicit_threshold


def evaluate_and_time(model, name: str) -> dict:
    """Fit model, compute ranking metrics + coverage, return dict."""
    t0 = time.time()
    model.fit(implicit_matrix)
    train_time = time.time() - t0

    metrics = evaluate_model(
        model=model,
        test_ratings=val_df,
        train_interaction=implicit_matrix,
        k=K,
        threshold=THRESHOLD,
    )

    sample_uids = [
        uid for uid in val_df["user_idx"].unique()[:SAMPLE_USERS] if uid < implicit_matrix.shape[0]
    ]
    all_recs = [
        [item_id for item_id, _ in model.recommend(int(uid), n=K, exclude_seen=True)]
        for uid in sample_uids
    ]
    metrics["coverage"] = coverage(all_recs, n_items)
    metrics["training_time_s"] = round(train_time, 2)

    print(
        f"[{name:15s}] {train_time:5.1f}s — P@{K}={metrics['precision_at_k']:.4f}  NDCG={metrics['ndcg_at_k']:.4f}"
    )
    return metrics


results: dict = {}

In [ ]:
results["Random"] = evaluate_and_time(RandomRecommender(), "Random")
results["Popularity"] = evaluate_and_time(PopularityRecommender(), "Popularity")

In [ ]:
als_model = ALSRecommender(
    factors=settings.als_factors,
    regularization=settings.als_regularization,
    iterations=settings.als_iterations,
    alpha=settings.als_alpha,
)
results["ALS"] = evaluate_and_time(als_model, "ALS")

In [ ]:
item_feat_df = build_item_features(train_df, movies_df)
genre_cols = [
    c
    for c in item_feat_df.columns
    if c not in ("avg_rating", "num_ratings", "popularity_score", "year")
    and not c.startswith("tag_tfidf_")
]
item_features_array = item_feat_df[genre_cols].values.astype(np.float32)

cb_model = ContentBasedRecommender()
cb_model.fit(explicit_matrix, item_features=item_features_array)
results["ContentBased"] = evaluate_and_time(cb_model, "ContentBased")

## 3. Results Table

In [ ]:
display_cols = ["precision_at_k", "recall_at_k", "ndcg_at_k", "coverage", "training_time_s"]
metrics_df = pd.DataFrame(results).T[display_cols]
metrics_df.columns = [f"P@{K}", f"R@{K}", f"NDCG@{K}", "Coverage", "Train(s)"]
metrics_df.index.name = "Model"

print(metrics_df.sort_values(f"P@{K}", ascending=False).to_string(float_format="{:.4f}".format))

## 4. Visualization

In [ ]:
model_names = list(results.keys())
COLORS = ["#6baed6", "#fd8d3c", "#74c476", "#9e9ac8"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f"Ranking Metrics @ K={K}", fontsize=14, fontweight="bold")

for ax, (key, label) in zip(
    axes,
    [
        ("precision_at_k", f"Precision@{K}"),
        ("recall_at_k", f"Recall@{K}"),
        ("ndcg_at_k", f"NDCG@{K}"),
    ],
    strict=False,
):
    values = [results[m][key] for m in model_names]
    bars = ax.bar(model_names, values, color=COLORS, edgecolor="white", linewidth=0.8)
    ax.set_title(label, fontsize=12)
    ax.set_ylabel("Score")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    ax.tick_params(axis="x", rotation=15)
    for bar, val in zip(bars, values, strict=False):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.01,
            f"{val:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.savefig("model_comparison_ranking.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle("Coverage & Training Time", fontsize=14, fontweight="bold")

cov_vals = [results[m]["coverage"] for m in model_names]
time_vals = [results[m]["training_time_s"] for m in model_names]

bars = axes[0].bar(model_names, cov_vals, color=COLORS, edgecolor="white")
axes[0].set_title("Catalog Coverage")
axes[0].set_ylabel("Fraction of catalog")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].tick_params(axis="x", rotation=15)
for bar, val in zip(bars, cov_vals, strict=False):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{val:.2%}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

bars = axes[1].bar(model_names, time_vals, color=COLORS, edgecolor="white")
axes[1].set_title("Training Time (s)")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=15)
for bar, val in zip(bars, time_vals, strict=False):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(time_vals) * 0.01,
        f"{val:.1f}s",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig("model_comparison_cost.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. ALS Hyperparameter Tuning

In [ ]:
factors_grid = [50, 100, 200]
alpha_grid = [20, 40, 80]
hp_results = []

for factors in factors_grid:
    for alpha in alpha_grid:
        m = ALSRecommender(
            factors=factors,
            regularization=settings.als_regularization,
            iterations=15,
            alpha=alpha,
        )
        t0 = time.time()
        m.fit(implicit_matrix)
        train_time = time.time() - t0
        met = evaluate_model(
            model=m,
            test_ratings=val_df,
            train_interaction=implicit_matrix,
            k=K,
            threshold=THRESHOLD,
        )
        hp_results.append(
            {
                "factors": factors,
                "alpha": alpha,
                "precision": met["precision_at_k"],
                "ndcg": met["ndcg_at_k"],
                "train_time_s": round(train_time, 1),
            }
        )
        print(
            f"factors={factors:3d}, alpha={alpha:3d} → P@{K}={met['precision_at_k']:.4f}, NDCG={met['ndcg_at_k']:.4f}  ({train_time:.1f}s)"
        )

hp_df = pd.DataFrame(hp_results)
print("\nBest 3 by Precision@K:")
print(hp_df.sort_values("precision", ascending=False).head(3).to_string(index=False))

In [ ]:
pivot = hp_df.pivot(index="factors", columns="alpha", values="precision")

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap="YlGn", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"alpha={a}" for a in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"factors={f}" for f in pivot.index])
ax.set_title(f"ALS Hyperparameter Grid — Precision@{K}", fontweight="bold")
plt.colorbar(im, ax=ax)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.4f}", ha="center", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("als_hyperparam_grid.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Sample Recommendations (ALS)

In [ ]:
DEMO_USER_IDX = 0
idx_to_title = movies_df.set_index("movie_idx")["title"].to_dict()

recs = als_model.recommend(DEMO_USER_IDX, n=10, exclude_seen=True)
print(f"Top-10 ALS recommendations for user_idx={DEMO_USER_IDX}:\n")
for rank, (movie_idx, score) in enumerate(recs, 1):
    title = idx_to_title.get(movie_idx, f"Movie {movie_idx}")
    print(f"  {rank:2d}. [{score:6.3f}] {title}")

## 7. Similar Items (ALS)

In [ ]:
toy_story = movies_df[movies_df["title"].str.contains("Toy Story", case=False, na=False)]
if toy_story.empty:
    SEED_ITEM = 0
    print("Toy Story not found, using movie_idx=0")
else:
    SEED_ITEM = int(toy_story.iloc[0]["movie_idx"])
    print(f"Seed: {toy_story.iloc[0]['title']} (movie_idx={SEED_ITEM})")

print("\nSimilar movies (ALS cosine):\n")
for rank, (movie_idx, score) in enumerate(als_model.similar_items(SEED_ITEM, n=10), 1):
    title = idx_to_title.get(movie_idx, f"Movie {movie_idx}")
    print(f"  {rank:2d}. [{score:.4f}] {title}")

## 8. Test Set Evaluation (Best Model)

In [ ]:
# Evaluate the best model (ALS) on the held-out test set
test_metrics = evaluate_model(
    model=als_model,
    test_ratings=test_df,
    train_interaction=implicit_matrix,
    k=K,
    threshold=THRESHOLD,
)
print("ALS — Test Set Results")
print(f"  Precision@{K} : {test_metrics['precision_at_k']:.4f}")
print(f"  Recall@{K}    : {test_metrics['recall_at_k']:.4f}")
print(f"  NDCG@{K}      : {test_metrics['ndcg_at_k']:.4f}")
print(f"  Users eval    : {int(test_metrics['n_users_evaluated']):,}")

## 9. Final Summary

In [ ]:
summary = metrics_df.sort_values(f"P@{K}", ascending=False)
best = summary.index[0]

print("=" * 65)
print(f"  Model Comparison — Validation Set (K={K})")
print("=" * 65)
print(summary.to_string(float_format="{:.4f}".format))
print("=" * 65)
print(f"  Best model : {best}")
print(f"  Test P@{K}  : {test_metrics['precision_at_k']:.4f}  (ALS)")
print("=" * 65)

## 10. Log Best Run to MLflow

Cell tùy chọn — chạy để track thí nghiệm lên MLflow UI.
Đảm bảo MLflow server đang chạy (`mlflow ui` hoặc qua docker-compose).

In [ ]:
import subprocess

import joblib

try:
    import mlflow
except ImportError:
    subprocess.check_call(["pip", "install", "mlflow", "-q"])
    import mlflow

# Best hyperparams from grid search
best_row = hp_df.sort_values("precision", ascending=False).iloc[0]
best_factors = int(best_row["factors"])
best_alpha = float(best_row["alpha"])

print(f"Best HP: factors={best_factors}, alpha={best_alpha}")

# Retrain with best HP on full train set
best_als = ALSRecommender(
    factors=best_factors,
    regularization=settings.als_regularization,
    iterations=settings.als_iterations,
    alpha=best_alpha,
)
best_als.fit(implicit_matrix)
best_test = evaluate_model(
    model=best_als,
    test_ratings=test_df,
    train_interaction=implicit_matrix,
    k=K,
    threshold=THRESHOLD,
)

# --- MLflow logging ---
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)

with mlflow.start_run(run_name="notebook_als_best") as run:
    mlflow.set_tag("source", "03_model_experiments.ipynb")
    mlflow.set_tag("model_type", "als")
    mlflow.set_tag("dataset_version", settings.dataset_name)

    mlflow.log_params(
        {
            "factors": best_factors,
            "alpha": best_alpha,
            "regularization": settings.als_regularization,
            "iterations": settings.als_iterations,
            "top_k": K,
        }
    )

    # Log all 4 model val metrics for comparison
    for model_name, res in results.items():
        prefix = model_name.lower()
        mlflow.log_metric(f"{prefix}_val_precision", res["precision_at_k"])
        mlflow.log_metric(f"{prefix}_val_ndcg", res["ndcg_at_k"])

    # Log best ALS test metrics
    mlflow.log_metrics(
        {
            "test_precision_at_k": best_test["precision_at_k"],
            "test_recall_at_k": best_test["recall_at_k"],
            "test_ndcg_at_k": best_test["ndcg_at_k"],
        }
    )

    # Save artifact
    import joblib

    artifact_path = PROJECT_ROOT / "artifacts" / "model_notebook.pkl"
    artifact_path.parent.mkdir(exist_ok=True)
    joblib.dump(
        {
            "model_type": "als",
            "user_factors": best_als.user_factors,
            "item_factors": best_als.item_factors,
            "n_users": best_als.user_factors.shape[0],
            "n_items": best_als.item_factors.shape[0],
            "hyperparams": {"factors": best_factors, "alpha": best_alpha},
            "metrics": best_test,
            "model_version": settings.model_version,
        },
        artifact_path,
    )
    mlflow.log_artifact(str(artifact_path))

    print(f"MLflow run logged: {run.info.run_id}")
    print(f"Test Precision@{K}: {best_test['precision_at_k']:.4f}")
    print(f"Tracking URI      : {settings.mlflow_tracking_uri}")